# 2.1 数据操作 (Data Manipulation)

## Intro

PyTorch's `Tensor`, TensorFlow's `Tensor`, and MXNet's `ndarray` are functionally the same concept as NumPy's `ndarray`, plus two extras: GPU acceleration and automatic differentiation. `jax.numpy.ndarray` follows the same pattern. Same base data structure, different superpowers — general framework mental model.

## 2.1.1 入门

In [ ]:
import torch
x = torch.arange(12)
x

Creates a 1-D tensor `[0..11]`, default dtype `int64`.

- Specify dtype up front to skip a later cast: `torch.arange(12, dtype=torch.float32)`
- `torch.arange` supports `(start, end, step)` like Python's `range`: `torch.arange(2, 10, 2)`

In [ ]:
x.shape       # torch.Size([12])
x.numel()     # 12

- `x.shape` ≡ `x.size()`
- Number of axes: `x.dim()` or `x.ndim`

In [ ]:
X = x.reshape(3, 4)
X

- `reshape` returns a view sharing memory when possible; copies only if data isn't contiguous
- `.view()` does the same but requires contiguity, errors otherwise — `reshape` is the safer default
- Auto-infer one dim: `x.reshape(-1, 4)` or `x.reshape(3, -1)`

## Tensor initialization

In [ ]:
torch.zeros((2, 3, 4))
torch.ones((2, 3, 4))
torch.randn(3, 4)          # standard normal
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

- `torch.full((shape), value)` — any constant fill
- `torch.eye(n)` — identity matrix
- `torch.rand(3, 4)` — uniform [0, 1)
- `torch.normal(mean, std, size=(3, 4))` — custom-parameter normal distribution
- `torch.manual_seed(0)` — reproducible random runs

## 2.1.2 运算符

In [ ]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y   # elementwise
torch.exp(x)                          # unary

- Operators are sugar for functions: `x + y` ≡ `torch.add(x, y)`
- Function form accepts `out=` to write into a pre-allocated tensor
- Other unary ops: `torch.log`, `torch.sqrt`, `torch.sin`

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0)   # stack rows → grows axis 0
torch.cat((X, Y), dim=1)   # stack columns → grows axis 1

- `torch.cat` joins along an existing axis, output has same ndim as inputs
- `torch.stack((X, Y))` creates a new axis instead — keeps X and Y as separate layers

In [ ]:
X == Y        # elementwise bool tensor
X.sum()       # scalar, sums everything

- `X.sum(dim=0)` — sum down each column
- `X.sum(dim=1, keepdim=True)` — keeps size-1 axis instead of squeezing, useful for broadcasting back against original shape
- Same axis/keepdim pattern applies to `.mean()`, `.max()`, `.min()`

## 2.1.3 广播机制 (Broadcasting)

In [ ]:
a = torch.arange(3).reshape((3, 1))   # 3×1
b = torch.arange(2).reshape((1, 2))   # 1×2
a + b                                  # broadcasts to 3×2

- Rule: shapes align from the rightmost dimension; two dims are compatible if equal or one of them is 1
- `a.expand(3, 2)` — preview a broadcasted view without performing the operation, useful for debugging shape mismatches

## 2.1.4 索引和切片

In [ ]:
X[-1], X[1:3]

In [ ]:
X[1, 2] = 9
X

In [ ]:
X[0:2, :] = 12
X

- Reverse: `X[::-1]` (mirrors Python list slicing; negative strides may be unsupported on GPU)
- Fancy indexing: `X[[0, 2]]` — grabs rows 0 and 2 directly, no loop
- Boolean masking: `X[X > 5]` — returns flat 1-D tensor of elements satisfying the condition
- Ellipsis: `X[..., 0]` — grabs index 0 of the last axis regardless of total ndim

## 2.1.5 节省内存

In [ ]:
before = id(Y)
Y = Y + X
id(Y) == before     # False — new memory allocated

In [ ]:
Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y         # in-place, memory reused
print('id(Z):', id(Z))

- Idiomatic in-place form: `X.add_(Y)` ≡ `X[:] = X + Y` (trailing underscore = in-place convention: `.mul_()`, `.clamp_()`, etc.)
- Caveat: in-place ops (`+=`, `X[:] = ...`, `.add_()`) can break autograd on leaf tensors with `requires_grad=True`, since PyTorch needs original values for gradient computation. Relevant once training starts.

## 2.1.6 转换为其他Python对象

In [ ]:
A = X.numpy()
B = torch.tensor(A)
type(A), type(B)

- `X.numpy()` shares memory with X
- `torch.tensor(A)` always copies — does NOT share memory with A
- To share memory from a NumPy array instead: `torch.from_numpy(A)`

In [ ]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

- `.item()` only works on single-element tensors
- For multi-element tensors as plain Python data: `.tolist()`